<a href="https://colab.research.google.com/github/donggeunlee77/MONAI-tutorial-exercise/blob/main/MONAI-2D-Segmentation-tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# unet_training_dict.py 파일로 연습

In [ ]:
import logging ## 로그 기록 및 출력하는 것. 학습 진행 상황이나 오류 출력용
import os ## 운영체제와 파일 시스템 조작.
import sys ## 파이썬 실행환경 제어
import tempfile ## 임시파일 및 폴더 생성
from glob import glob ## 특정 패턴의 파일 검색 - 예를들면 glob("경로/*.png") 이렇게 하면 해당 경로 폴더 안에 있는 모든 PNG 파일의 경로를 리스트로 가져와줌.

import torch ## 파이토치 핵심라이브러리
from PIL import Image ## 이미지파일 읽고 처리하는 라이브러리 (Python Imaging Library)
from torch.utils.tensorboard import SummaryWriter

import monai
from monai.data import create_test_image_2d, list_data_collate, decollate_batch, DataLoader ## 순서대로 테스트용 2D 이미지와 정답마크 생성 / 여러 데이터 샘플을 하나의 Batch로 결합 / Batch를 개별샘플로 분리 / 데이터를 Batch 단위로 모델에 공급
from monai.inferers import sliding_window_inference ## 슬라이딩 윈도우 추론 : 큰 이미지를 한번에 모델에 입력하는 대신 작은 영역으로 나누어 순차적으로 예측한 다음 결과를 합치는 방식. GPU 메모리 문제로.
from monai.metrics import DiceMetric ## Segmentation 성능을 평가하는 Dice score : 예측된 영역과 실제 정답 영역이 얼마나 겹치는지
from monai.transforms import (
    Activations,
    EnsureChannelFirstd,
    AsDiscrete, ## 연속적인 출력을 이산적인 label로 변환
    Compose,
    LoadImaged,
    RandCropByPosNegLabeld, ##정답 영역의 양성, 음성 비율을 고려하여 이미지 crop하는 것. 전체 영상에서 병변이 차지하는 비율이 매우 적을 수 있는데, 무작위로 crop하면 병변이 없는 영역만 계속 학습할수도 있으니.
    RandRotate90d,
    ScaleIntensityd ##이미지 밝기 및 intensity 정규화
) ## d 는 dictionary 기반 transform을 의미함. 파이썬 딕셔너리(Dictionary)는 키(Key)와 값(Value)의 쌍을 한 쌍으로 갖는 자료형. 예를들어 {"img" : "image001.png", "seg" : "label001.png"}, user = {"name": "홍길동", "age": 25, "email": "hong@example.com"}
from monai.visualize import plot_2d_or_3d_image ## 2D또는 3D이미지를 시각화하는 기능. segmentation에서는 모델이 실제로 병변을 잘 찾아내고 있는지 눈으로 확인하는 것이 중요하기 떄문에 자주 활용됨.


이후, 전체 구조는 아래와 같은 구조로 이루어짐
1) 프로그램 실행 (if __name__ == "__main__": )
2) 임시 데이터 저장 공간 생성 (tempfile.TemporaryDictionary())
3) main(tempdir) 실행

main() 내부에서는
(1) 가상 영상 및 정답 mask 생성
(2) train/validation 데이터 분리
(3) 이미지 전처리(transform)
(4) Dataset 및 DataLoader 생성
(5) U-Net 모델 및 Loss, Optimizer 설정
(6) 모델 학습(training)
(7) validation 및 dice 평가
(8) 최적 모델 저장 및 Tensorboard 기록
의 과정이 일어남. 즉 main(0은 이 모든 과정을 순서대로 실행하도록 묶어놓은 함수.

main은 특별한 예약어가 아니라 개발자가 임의로 붙인 함수 이름. 예를 들어
def main():
 print("Hello")
 print("MONAI")

main()
을 실행하면
Hello
MONAI
이렇게 출력됨.

근데 우리 코드는
def main(tempdir): 이렇게 괄호 안에 변수가 들어있음. 즉 이는 함수를 실행할 때 외부에서 데이터를 전달받겠다는 의미.
글고 if __name__ == "__main__"는 이 파일을 직접 실행했을 때만 아래 코드를 실행하는 것이라고 하네

In [ ]:
def main(tempdir):
  monai.config.print_config()  # 현재 설치된 MONAI 및 관련 라이브러리의 버전과 환경 출력
  logging.basicConfig(stream=sys.stdout, level=logging.INFO) ## 프로그램 실행 중 발생하는 로그를 설정 // 이까지는 학습에 들어가기 전 실행 환경을 확인하는 과정.

  ## create a temporary dictionary and 40 random image, mask pairs
  print(f"generating synthetic data to {tempdir} (this may take a while)")
  for i in range(40):
    im, seg = create_test_image_2d(128, 128, num_seg_classes=1) ## 원본이미지 im과 정답 segmentation mask인 seg 가 생성됨. for 구문에 따라 총 40개겠지?
    Image.fromarray((im * 255).astype("uint8")).save(os.path.join(tempdir, f"img{i:d}.png")) ## im 혹은 seg * 255는 0~1범위의 픽셀값을 0~255로 변환, astype("uint8")은 8bit 정수형으로 변환, Image.fromarray는 배열을 이미지 객체로 변환.
    Image.fromarray((seg * 255).astype("uint8")).save(os.path.join(tempdir, f"seg{i:d}.png")) ## 결과적으로 임시폴더에 img0.png ~ img39.png / seg0.png~seg39.png 가 생성됨.

  images = sorted(glob(os.path.join(tempdir, "img*.png"))) ## glob은 특정 패턴에 해당하는 파일 검색, sorted는 파일 정렬
  segs = sorted(glob(os.path.join(tempdir, "seg*.png")))
  train_files = [{"img" : img, "seg" : seg} for img, seg in zip(images[:20], segs[:20])] ## zip은 iterable한 객체 여러개를 받아 동일한 인덱스의 요소를 튜플로 묶어주는 반복자(iterator)를 반환해줌. 예를들어 numbers = [1,2,3], letters=['A','B','C'] 이면 zip(numbers, letters)는 [(1, 'A'), (2, 'B'), (3, 'C')] 이렇게 나오는것
  val_files = [{"img" : img, "seg" : seg} for img, seg in zip(images[-20:], segs[-20:])] ## train_files는 앞에서 20개, val_files는 뒤에서 20개

  ## define transforms for image and segmentation
  train_transforms = Compose(
      [
          LoadImaged(keys=["img", "seg"]),
          EnsureChannelFirstd(keys=["img", "seg"]),
          ScaleIntensityd(keys=["img", "seg"]), ## 픽셀 intensity의 정규화
          RandCropByPosNegLabeld(  ## 정답 mask 를 기준으로 무작위 cropd
              keys=["img", "seg"], label_key="seg", spatial_size=[96,96], pos=1, neg=1, num_samples=4
          ), ## 원본영상은 128*128이지만, 학습시에는 96*96 크기로 잘라서 사용하겠다. 그리고 정답 mask를 기준으로 양성 영역과 음성 영역을 고려해 crop 하겠다. pos=1과 neg=1은 각각 영역을 포함하는 crop의 선택 가중치, num_samples=4는 이미지 하나당 4개의 crop 생성.
          RandRotate90d(keys=["img", "seg"], prob=0.5, spatial_axes=[0,1]) ## 핵심은, 원본 영상과 정답 mask를 동시에 회전시킴.
      ]
  )
  val_transforms = Compose(
      [
          LoadImaged(keys=["img", "seg"]),
          EnsureChannelFirstd(keys=["img", "seg"]),
          ScaleIntensityd(keys=["img", "seg"])
      ]
  ) # validation은 crop이나 회전 안하고 일관된 조건에서 수행.

  ## define dataset and data loader
  check_ds = monai.data.Dataset(data=train_files, transform=train_transforms)  ## Dataset은 학습에 사용할 데이터와 전처리 과정을 묶어놓은 객체야.
  ## use batch_size=2 to load images and use RandCropByPosNegLabeld to generate 2 x 4 images for network training
  check_loader = DataLoader(check_ds, batch_size=2, num_workers=4, collate_fn=list_data_collate) ## collate_fn 은 여러 데이터를 하나의 batch로 묶는 방식. 이때 위에 한 이미지에서 4개의 crop 이미지가 나오니까, 최종적으로 한 batch당 모델에 입력되는 영상은 8개.
  check_data = monai.utils.misc.first(check_loader) ## 첫번째 batch를 확인하는 용도의 코드 . 이 check_ds와 check_loader는 데이터 형태를 확인하기 위한 별도의 객체
  print(check_data["img"].shape, check_data["seg"].shape) ## 예상되는 shape는 (8, 1, 96, 96), (8, 1, 96, 96)

  ## create a training data loader
  train_ds = monai.data.Dataset(data=train_files, transform=train_transforms)
  train_loader = DataLoader(train_ds, batch_size=2, num_workers=4, shuffle=True, collate_fn=list_data_collate, pin_memory=torch.cuda.is_available()) ## shuffle은 매 epoch마다 데이터 순서 섞기, pin_memory는 GPU 학습시 데이터 전송 효율을 높일 수 있도록 메모리 설정
  ## create a validation data loader
  val_ds = monai.data.Dataset(data=val_files, transform=val_transforms)
  val_loader = DataLoader(val_ds, batch_size=1, num_workers=4, collate_fn=list_data_collate)
  dice_metric = DiceMetric(include_background=True, reduction="mean", get_non_nans=False) ##Dice score 계산용
  post_trans = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)]) ##후처리. Activations(sigmoid=True)는 모델이 출력한 값을 0-1사이 확률값으로 변환, AsDiscrete(~)는 확률이 0.5 이상이면 1, 미만이면 0으로 변환 -> 최종적으로 Binary Segmentation Mask를 만드는 것.

  ## create UNet, DiceLoss and Adam optimizer
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model = monai.networks.nets.UNet(
      spatial_dims=2, ##2D라고
      in_channels=1, ##입력 채널 수
      out_channels=1, ##출력 채널 수. 즉 binary segmentation을 수행한다는 뜻. 다중장기 분할 원하면 이거 수 변
      channels=(16, 32, 64, 128, 256), ##Unet 각 단계의 feature channel 수
      strides=(2,2,2,2), ##각 단계에서 공간 해상도를 줄이는 정도
      num_res_units=2 ##각 단계의 residual unit 구성)
  ).to(device)
  loss_function = monai.losses.DiceLoss(sigmoid=True) ## Dice Loss : 모델의 예측 결과와 정답 mask간의 차이를 계산하는 Loss function.
  optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

  ## start a typical PyTorch training
  val_interval = 2
  best_metrics = -1
  best_metric_epoch = -1
  epoch_loss_values = list()
  metric_values = list()
  writer = SummaryWriter()
  for epoch in range(10):
    print("-" * 10)
    print(f"epoch : {epoch + 1}/10")
    model.train()
    epoch_loss=0
    step=0
    for batch_data in train_loader:
      step += 1
      inputs, labels= batch_data["img"].to(device), batch_data["seg"].to(device)
      optimizer.zero_grad() ## 이전 batch의 gradient 초기화
      outputs = model(inputs) ## 모델에 영상 입력
      loss = loss_function(outputs, labels) ## 예측과 정답 비교하여 DIceLoss 계산
      loss.backward()
      optimizer.step() ## 여기서 parameter 업데이트
      epoch_loss += loss.item() ## batch별 loss를 축적
      epoch_len = len(train_ds) // train_loader.batch_size ## 1에포크당 길이 계산하기. train_ds 길이를 배치의 크기로 나누면 1 epoch는 몇 batch인지, 즉 총 몇 step이 걸리는지 나오는것. // 는 나머지 버리고 몫만 반환해줌.
      print(f"{step}/{epoch_len}, train_loss={loss.iten():.4f}")
      writer.add_scalar("train_loss", loss.item(), epoch_len * epoch + step) ## batch당 loss를 기록
    epoch_loss /= step ## batch 다 돌렸으면 batch별 loss 평균내기
    epoch_loss_values.append(epoch_loss)  ## 평균낸 해당 epoch의 epoch_loss를 리스트에 입력해줌
    print(f"epoch {epoch +1} average loss : {epoch_loss:.4f}")

    if (epoch + 1) % val_interval == 0:
      model.eval()
      with torch.no_grad():
        val_images = None
        val_labels = None
        val_outputs = None
        for val_data in val_loader:
          val_images, val_labels = val_data["img"].to(device), val_data["seg"].to(device)
          roi_size = (96, 96)
          sw_batch_size = 4
          val_outputs = sliding_window_inference(val_images, roi_size, sw_batch_size, model)
          val_outputs = [post_trans(i) for i in decollate_batch(val_outputs)] ## Sliding Window Inference는 큰 영상을 일정한 크기의 작은 영역으로 나누어 추론한 다음 결과를 합치는 방식이야. 이번 예제에서는 Validation 영상이 128 × 128이고, 모델의 추론 영역은 96 × 96이야. 따라서 전체 영상을 한 번에 처리하는 대신 96 × 96 크기의 Window를 이용해 추론하는 거야. 이 방식은 특히 CT, MRI처럼 해상도가 큰 영상에서 GPU 메모리 사용량을 조절하는 데 유용해.
          # compute metric for current iteration
          dice_metric(y_pred = val_outputs, y=val_labels)
        # aggregate the final mean dice result


